# 02 — Final YOLO Training, Controlled Experiments and Model Selection

## Number Plate Detection of Vehicles

This notebook documents the **final detector-training and model-selection process** for the project using Google Colab GPU.

The analysis is deliberately **result-driven and anti-regression focused**. A frozen dataset of **115 images and 233 annotated number-plate boxes** is used with the fixed **90/13/12 train-validation-test split** established during EDA.

The notebook evaluates four detector configurations:

1. **Existing `best.pt` checkpoint** — baseline / incumbent model.
2. **YOLO26s @ 832 px** — controlled retraining experiment.
3. **YOLO26s @ 1024 px** — higher-resolution retraining experiment.
4. **YOLO26m @ 832 px** — higher-capacity supplementary experiment.

Model selection is based on **validation mAP@50–95**, not on test-set performance. The untouched test set is used only for the selected final model.

> **Important:** The project brief mentions a 95%+ detection target, but object detection does not have one universal “accuracy” measure. This notebook therefore reports standard detector metrics: Precision, Recall, mAP@50 and mAP@50–95.


## 0. Files required in Google Drive

Create a ZIP named:

`number_plate_training.zip`

with this structure:

```text
number_plate_training/
├── data/
│   └── processed/
│       ├── train/
│       │   ├── images/
│       │   └── labels/
│       ├── val/
│       │   ├── images/
│       │   └── labels/
│       └── test/
│           ├── images/
│           └── labels/
└── models/
    └── best.pt        # optional but strongly recommended: current deployed model
```

Upload the ZIP to the **root of My Drive**.

The extraction cell below normalizes both supported ZIP layouts:

- `number_plate_training/data/...`
- `data/...`

In either case, the final Colab project root is always:

`/content/number_plate_training`

You do **not** need to upload `.venv/`, previous `runs/`, reports, OCR crops, Streamlit files, old ZIPs, or `__pycache__/` directories. The notebook creates its own YAML configuration and output folders.


## 1. Verify the GPU

The project specifically requires GPU-accelerated training.  
This cell stops immediately if CUDA is unavailable so a long CPU training run cannot start accidentally.

In [1]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), (
    "CUDA GPU is not available. In Colab choose Runtime > Change runtime type > T4 GPU."
)

GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)
print("CUDA available:", torch.cuda.is_available())

Thu Sep  3 10:21:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Google Drive and extract the frozen dataset

Training is performed in `/content/` because it is faster than training directly from Drive.  
Only final checkpoints, tables and selected results are copied back to Drive.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from zipfile import ZipFile
import shutil

ZIP_PATH = Path("/content/drive/MyDrive/number_plate_training.zip")
PROJECT_DIR = Path("/content/number_plate_training")
TEMP_EXTRACT = Path("/content/_number_plate_training_extract")
DRIVE_OUTPUT = Path("/content/drive/MyDrive/Number-Plate-Detection-Training-Results")

assert ZIP_PATH.exists(), f"Upload {ZIP_PATH.name} to the root of My Drive first."

# Start from a clean local workspace.
for path in [PROJECT_DIR, TEMP_EXTRACT]:
    if path.exists():
        shutil.rmtree(path)

TEMP_EXTRACT.mkdir(parents=True, exist_ok=True)

with ZipFile(ZIP_PATH, "r") as z:
    z.extractall(TEMP_EXTRACT)

# Support both ZIP layouts:
#   number_plate_training/data/...  OR  data/...
nested_root = TEMP_EXTRACT / "number_plate_training"
if (nested_root / "data" / "processed").exists():
    extracted_root = nested_root
elif (TEMP_EXTRACT / "data" / "processed").exists():
    extracted_root = TEMP_EXTRACT
else:
    raise FileNotFoundError(
        "Could not find data/processed in number_plate_training.zip. "
        "Check the ZIP folder structure shown above."
    )

# Normalize the extracted project to exactly /content/number_plate_training.
shutil.copytree(extracted_root, PROJECT_DIR)
shutil.rmtree(TEMP_EXTRACT)

DATA_DIR = PROJECT_DIR / "data" / "processed"
MODELS_DIR = PROJECT_DIR / "models"
CONFIG_DIR = PROJECT_DIR / "configs"

assert DATA_DIR.exists(), f"Processed dataset not found: {DATA_DIR}"

for split in ["train", "val", "test"]:
    assert (DATA_DIR / split / "images").exists(), f"Missing {split}/images"
    assert (DATA_DIR / split / "labels").exists(), f"Missing {split}/labels"

print("Project directory :", PROJECT_DIR)
print("Dataset directory :", DATA_DIR)
print("Models directory  :", MODELS_DIR)

assert str(PROJECT_DIR) == "/content/number_plate_training"
assert "number_plate_training/number_plate_training" not in str(PROJECT_DIR)

print("\n✅ ZIP extracted and project path normalized successfully.")


Mounted at /content/drive
Project directory : /content/number_plate_training
Dataset directory : /content/number_plate_training/data/processed
Models directory  : /content/number_plate_training/models

✅ ZIP extracted and project path normalized successfully.


## 3. Install the training framework and validate the dataset

The final EDA reported:

- **115 images**
- **90 train / 13 validation / 12 test**
- **233 number-plate boxes**
- **0 invalid YOLO annotations**

This cell uses Colab's compatible Pandas version and independently recounts the uploaded package so training cannot accidentally proceed with a different dataset.


In [3]:
# Install only the packages required for this notebook.
# Pin Pandas to Colab's compatible release instead of upgrading to Pandas 3.x.
!pip -q install -U ultralytics pyyaml "pandas==2.2.3"

from pathlib import Path
import pandas as pd
import yaml
import ultralytics

print("Ultralytics:", ultralytics.__version__)
print("Pandas      :", pd.__version__)

assert pd.__version__ == "2.2.3", (
    f"Expected pandas 2.2.3 in this Colab notebook, found {pd.__version__}. "
    "If this runtime previously loaded another Pandas version, restart the runtime and run from the top."
)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

rows = []
for split in ["train", "val", "test"]:
    image_dir = DATA_DIR / split / "images"
    label_dir = DATA_DIR / split / "labels"

    images = sorted(
        p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )
    labels = sorted(label_dir.glob("*.txt"))

    box_count = 0
    for label_path in labels:
        box_count += sum(
            1 for line in label_path.read_text().splitlines()
            if line.strip()
        )

    rows.append({
        "split": split,
        "images": len(images),
        "label_files": len(labels),
        "boxes": box_count,
    })

dataset_check = pd.DataFrame(rows)
display(dataset_check)

# Frozen-dataset checks from the final EDA.
assert dataset_check["images"].sum() == 115, (
    f"Expected 115 images, found {dataset_check['images'].sum()}."
)
assert dataset_check["boxes"].sum() == 233, (
    f"Expected 233 boxes, found {dataset_check['boxes'].sum()}."
)

split_check = dataset_check.set_index("split")
assert split_check.loc["train", "images"] == 90, "Train split should contain 90 images."
assert split_check.loc["val", "images"] == 13, "Validation split should contain 13 images."
assert split_check.loc["test", "images"] == 12, "Test split should contain 12 images."

# Every image in this frozen dataset has a corresponding label file.
assert (dataset_check["images"] == dataset_check["label_files"]).all(), (
    "At least one split has a different number of images and label files."
)

CONFIG_DIR.mkdir(parents=True, exist_ok=True)

data_yaml = {
    "path": str(DATA_DIR),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": {0: "number_plate"},
}

DATA_YAML = CONFIG_DIR / "data_colab.yaml"
with DATA_YAML.open("w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print("\n✅ Dataset validation passed.")
print("✅ Total images :", int(dataset_check["images"].sum()))
print("✅ Total boxes  :", int(dataset_check["boxes"].sum()))
print("✅ YOLO YAML    :", DATA_YAML)

assert str(DATA_YAML) == "/content/number_plate_training/configs/data_colab.yaml"
print("✅ Project/YAML path is clean (no duplicated folder).")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 6.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics: 8.4.138
Pandas      : 2.2.3


,split,images,label_files,boxes
0,train,90,90,185
1,val,13,13,23
2,test,12,12,25



✅ Dataset validation passed.
✅ Total images : 115
✅ Total boxes  : 233
✅ YOLO YAML    : /content/number_plate_training/configs/data_colab.yaml
✅ Project/YAML path is clean (no duplicated folder).


## 4. Reproducibility and Experimental Strategy

Because the dataset is small and number plates can occupy a relatively small portion of high-resolution vehicle images, the experiment design focuses on **controlled changes rather than uncontrolled model switching**.

The main controls are:

- fixed train / validation / test split;
- fixed random seed;
- pretrained YOLO weights;
- AdamW optimization;
- early stopping;
- automatic batch sizing;
- identical augmentation settings where comparisons are intended to isolate model size or input resolution;
- model selection using **validation mAP@50–95**;
- untouched test evaluation only after selection.

### Primary Experiments

The first two retraining candidates use **YOLO26s** with identical optimization settings but different input resolutions:

- **YOLO26s @ 832 px**
- **YOLO26s @ 1024 px**

This tests whether increased image resolution improves localization of smaller plate regions.

### Supplementary Capacity Experiment

After the YOLO26s experiments, **YOLO26m @ 832 px** is tested separately to determine whether increasing model capacity improves recall/generalization.

The supplementary experiment is treated as a controlled architecture-capacity check. It is **not allowed to replace the final model unless it first beats the incumbent on the validation selection metric**.


In [5]:
from ultralytics import YOLO
import random
import numpy as np
import torch
import os
import pandas as pd
from IPython.display import display

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

RUN_ROOT = PROJECT_DIR / "runs" / "detect"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

# Explicit configurations make hyperparameter tuning visible and defensible.
CANDIDATES = [
    {
        "name": "yolo26s_832",
        "model": "yolo26s.pt",
        "imgsz": 832,
        "epochs": 120,
        "batch": -1,
        "optimizer": "AdamW",
        "lr0": 0.001,
        "lrf": 0.01,
        "weight_decay": 0.0005,
        "patience": 25,
        "degrees": 5.0,
        "translate": 0.10,
        "scale": 0.40,
        "fliplr": 0.50,
        "hsv_h": 0.015,
        "hsv_s": 0.60,
        "hsv_v": 0.35,
        "mosaic": 0.80,
        "mixup": 0.05,
        "erasing": 0.10,
    },
    {
        "name": "yolo26s_1024",
        "model": "yolo26s.pt",
        "imgsz": 1024,
        "epochs": 120,
        "batch": -1,
        "optimizer": "AdamW",
        "lr0": 0.001,
        "lrf": 0.01,
        "weight_decay": 0.0005,
        "patience": 25,
        "degrees": 5.0,
        "translate": 0.10,
        "scale": 0.40,
        "fliplr": 0.50,
        "hsv_h": 0.015,
        "hsv_s": 0.60,
        "hsv_v": 0.35,
        "mosaic": 0.80,
        "mixup": 0.05,
        "erasing": 0.10,
    },
]

display(pd.DataFrame(CANDIDATES))

,name,model,imgsz,epochs,batch,optimizer,lr0,lrf,weight_decay,patience,degrees,translate,scale,fliplr,hsv_h,hsv_s,hsv_v,mosaic,mixup,erasing
0,yolo26s_832,yolo26s.pt,832,120,-1,AdamW,0.001,0.01,0.0005,25,5.0,0.1,0.4,0.5,0.015,0.6,0.35,0.8,0.05,0.1
1,yolo26s_1024,yolo26s.pt,1024,120,-1,AdamW,0.001,0.01,0.0005,25,5.0,0.1,0.4,0.5,0.015,0.6,0.35,0.8,0.05,0.1


## 5. Evaluate the existing model on validation data

If `models/best.pt` is present in the uploaded ZIP, it is evaluated **before retraining** on the same validation split.

This gives us a fair “do not regress” baseline.  
If the new models perform worse, the existing detector remains the selected model.

In [6]:
validation_rows = []

EXISTING_MODEL = PROJECT_DIR / "models" / "best.pt"

def metric_row(name, model_path, imgsz, metrics, source):
    speed = getattr(metrics, "speed", {}) or {}
    return {
        "candidate": name,
        "source": source,
        "model_path": str(model_path),
        "imgsz": imgsz,
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "mAP50": float(metrics.box.map50),
        "mAP50_95": float(metrics.box.map),
        "preprocess_ms": float(speed.get("preprocess", float("nan"))),
        "inference_ms": float(speed.get("inference", float("nan"))),
        "postprocess_ms": float(speed.get("postprocess", float("nan"))),
    }

if EXISTING_MODEL.exists():
    print("Evaluating existing baseline:", EXISTING_MODEL)
    existing = YOLO(str(EXISTING_MODEL))
    existing_val = existing.val(
        data=str(DATA_YAML),
        split="val",
        imgsz=832,
        batch=4,
        device=0,
        plots=False,
        verbose=False,
    )
    validation_rows.append(
        metric_row("existing_best", EXISTING_MODEL, 832, existing_val, "existing")
    )
else:
    print("No existing models/best.pt uploaded. New candidates will be compared with each other only.")

pd.DataFrame(validation_rows)

Evaluating existing baseline: /content/number_plate_training/models/best.pt
Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s summary (fused): 122 layers, 9,465,567 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2782.6±1175.6 MB/s, size: 1536.1 KB)
val: Scanning /content/number_plate_training/data/processed/val/labels... 13 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 13/13 203.7it/s 0.1s
val: New cache created: /content/number_plate_training/data/processed/val/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.2it/s 1.9s
                   all         13         23      0.991       0.87      0.963      0.522
Speed: 1.8ms preprocess, 31.4ms inference, 0.0ms loss, 0.3ms postprocess per image


,candidate,source,model_path,imgsz,precision,recall,mAP50,mAP50_95,preprocess_ms,inference_ms,postprocess_ms
0,existing_best,existing,/content/number_plate_training/models/best.pt,832,0.991076,0.869565,0.963474,0.521626,1.80663,31.424725,0.265718


## 6. Train the YOLO26s candidates on GPU

Both candidates use the **same frozen training and validation sets**.

The only intentional architectural/training comparison is input resolution, while the main optimization settings are held constant.

Key tuned hyperparameters:

- **Optimizer:** AdamW
- **Initial learning rate (`lr0`):** 0.001
- **Final LR fraction (`lrf`):** 0.01
- **Automatic batch size:** `batch=-1`
- **Patience:** 25 epochs
- **Epoch ceiling:** 120
- **Input sizes:** 832 and 1024
- controlled HSV, rotation, translation, scaling, flipping, mosaic, mixup and erasing

The model's early-stopping mechanism can terminate training before 120 epochs when validation improvement stalls.

In [7]:
trained_models = {}

for cfg in CANDIDATES:
    print("\n" + "=" * 80)
    print("TRAINING:", cfg["name"])
    print("=" * 80)

    model = YOLO(cfg["model"])

    model.train(
        data=str(DATA_YAML),
        imgsz=cfg["imgsz"],
        epochs=cfg["epochs"],
        batch=cfg["batch"],
        optimizer=cfg["optimizer"],
        lr0=cfg["lr0"],
        lrf=cfg["lrf"],
        weight_decay=cfg["weight_decay"],
        device=0,
        workers=2,
        seed=SEED,
        deterministic=True,
        patience=cfg["patience"],
        project=str(RUN_ROOT),
        name=cfg["name"],
        exist_ok=True,
        plots=True,

        # Augmentation
        hsv_h=cfg["hsv_h"],
        hsv_s=cfg["hsv_s"],
        hsv_v=cfg["hsv_v"],
        degrees=cfg["degrees"],
        translate=cfg["translate"],
        scale=cfg["scale"],
        fliplr=cfg["fliplr"],
        mosaic=cfg["mosaic"],
        mixup=cfg["mixup"],
        erasing=cfg["erasing"],
    )

    best_path = RUN_ROOT / cfg["name"] / "weights" / "best.pt"
    assert best_path.exists(), f"Training finished but best.pt not found: {best_path}"

    trained_models[cfg["name"]] = best_path

print("\nTraining completed for:", list(trained_models))


TRAINING: yolo26s_832
Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/number_plate_training/configs/data_colab.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.1, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.6, hsv_v=0.35, imgsz=832, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=0.8, mul

## 7. Primary Validation Comparison and Model Selection

This is the first critical anti-regression step.

The existing checkpoint and both YOLO26s retraining candidates are evaluated on the **same frozen validation set**.

### Selection Rule

The model with the highest **validation mAP@50–95** is selected.

The test split is **not** consulted during this decision.

This prevents two common evaluation errors:

- replacing a strong existing model simply because a newer model was retrained;
- repeatedly checking the test set and selecting the model that happens to look best there.

The validation comparison below therefore determines whether either YOLO26s retraining experiment provides a real improvement.


In [8]:
for cfg in CANDIDATES:
    best_path = trained_models[cfg["name"]]
    candidate_model = YOLO(str(best_path))

    val_metrics = candidate_model.val(
        data=str(DATA_YAML),
        split="val",
        imgsz=cfg["imgsz"],
        batch=4,
        device=0,
        plots=False,
        verbose=False,
    )

    validation_rows.append(
        metric_row(cfg["name"], best_path, cfg["imgsz"], val_metrics, "retrained")
    )

comparison = pd.DataFrame(validation_rows)
comparison = comparison.sort_values(
    ["mAP50_95", "mAP50", "recall"],
    ascending=False
).reset_index(drop=True)

display(
    comparison[
        ["candidate", "source", "imgsz", "precision", "recall",
         "mAP50", "mAP50_95", "inference_ms"]
    ].style.format({
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "mAP50": "{:.4f}",
        "mAP50_95": "{:.4f}",
        "inference_ms": "{:.2f}",
    })
)

winner = comparison.iloc[0]
WINNER_NAME = winner["candidate"]
WINNER_PATH = Path(winner["model_path"])
WINNER_IMGSZ = int(winner["imgsz"])

print("\nSelected model:", WINNER_NAME)
print("Selected path :", WINNER_PATH)
print("Validation mAP50-95:", f"{winner['mAP50_95']:.4f}")
print("Validation mAP50   :", f"{winner['mAP50']:.4f}")

if "existing_best" in comparison["candidate"].values:
    old = comparison.loc[comparison["candidate"] == "existing_best"].iloc[0]
    print("\nExisting baseline mAP50-95:", f"{old['mAP50_95']:.4f}")
    print("Selected model mAP50-95 :", f"{winner['mAP50_95']:.4f}")
    print(
        "Change:",
        f"{(winner['mAP50_95'] - old['mAP50_95']) * 100:+.2f} percentage points"
    )

Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s summary (fused): 122 layers, 9,465,567 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4310.3±640.6 MB/s, size: 2125.4 KB)
val: Scanning /content/number_plate_training/data/processed/val/labels.cache... 13 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 13/13 3.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 4.0it/s 1.0s
                   all         13         23      0.902      0.797      0.878      0.482
Speed: 2.4ms preprocess, 16.8ms inference, 0.0ms loss, 0.6ms postprocess per image
Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s summary (fused): 122 layers, 9,465,567 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3885.3±255.3 MB/s, size: 1894.3 KB)
val: Scanning /content/number_plate_trai

,candidate,source,imgsz,precision,recall,mAP50,mAP50_95,inference_ms
0,existing_best,existing,832,0.9911,0.8696,0.9635,0.5216,31.42
1,yolo26s_832,retrained,832,0.9016,0.7970,0.8784,0.4824,16.81
2,yolo26s_1024,retrained,1024,0.9495,0.8181,0.8755,0.4667,44.72



Selected model: existing_best
Selected path : /content/number_plate_training/models/best.pt
Validation mAP50-95: 0.5216
Validation mAP50   : 0.9635

Existing baseline mAP50-95: 0.5216
Selected model mAP50-95 : 0.5216
Change: +0.00 percentage points


### Primary Experiment Result

The primary experiments did **not** outperform the existing checkpoint.

The retained baseline achieved approximately:

- **Validation Precision:** 0.9911
- **Validation Recall:** 0.8696
- **Validation mAP@50:** 0.9635
- **Validation mAP@50–95:** 0.5216

Both retrained YOLO26s candidates produced lower validation mAP@50–95:

- **YOLO26s @ 832 px:** ~0.482
- **YOLO26s @ 1024 px:** ~0.467

### Interpretation

Increasing the input resolution from 832 to 1024 did **not** improve the detector. The 1024 px experiment was also more computationally expensive.

Therefore, the existing checkpoint was retained rather than replacing it with a weaker retrained model.

This is an **anti-regression result**: retraining was useful because it demonstrated that the existing detector remained stronger under a controlled validation comparison.


## 8. Final Untouched Test-Set Evaluation

After validation-based model selection, only the selected **existing `best.pt` checkpoint** is evaluated on the untouched test set.

The test set contains:

- **12 images**
- **25 annotated number plates**

The final test metrics quantify generalization to unseen images.

The most important distinction is:

- **validation recall:** approximately **86.96%**
- **untouched test recall:** **60.00%**

The 60% value is therefore the final unseen-data recall of the selected detector—not the validation recall.


In [9]:
final_model = YOLO(str(WINNER_PATH))

test_metrics = final_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=WINNER_IMGSZ,
    batch=1,
    device=0,
    plots=True,
    project=str(PROJECT_DIR / "runs" / "test"),
    name="final_test",
    exist_ok=True,
)

speed = getattr(test_metrics, "speed", {}) or {}

final_test = pd.DataFrame([{
    "selected_model": WINNER_NAME,
    "imgsz": WINNER_IMGSZ,
    "precision": float(test_metrics.box.mp),
    "recall": float(test_metrics.box.mr),
    "mAP50": float(test_metrics.box.map50),
    "mAP50_95": float(test_metrics.box.map),
    "preprocess_ms": float(speed.get("preprocess", float("nan"))),
    "inference_ms": float(speed.get("inference", float("nan"))),
    "postprocess_ms": float(speed.get("postprocess", float("nan"))),
    "gpu": GPU_NAME,
}])

display(final_test.style.format({
    "precision": "{:.4f}",
    "recall": "{:.4f}",
    "mAP50": "{:.4f}",
    "mAP50_95": "{:.4f}",
    "preprocess_ms": "{:.2f}",
    "inference_ms": "{:.2f}",
    "postprocess_ms": "{:.2f}",
}))

Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s summary (fused): 122 layers, 9,465,567 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 89.6±19.8 MB/s, size: 2048.3 KB)
val: Scanning /content/number_plate_training/data/processed/test/labels... 12 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 12/12 383.8it/s 0.0s
val: New cache created: /content/number_plate_training/data/processed/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 11.0it/s 1.1s
                   all         12         25       0.93        0.6      0.686      0.383
Speed: 2.0ms preprocess, 36.6ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to /content/number_plate_training/runs/test/final_test


,selected_model,imgsz,precision,recall,mAP50,mAP50_95,preprocess_ms,inference_ms,postprocess_ms,gpu
0,existing_best,832,0.9305,0.6000,0.6862,0.3832,2.02,36.62,0.52,Tesla T4


### Final Test Result — What the Metrics Mean

The selected detector achieved:

| Metric | Final Test Result | Interpretation |
|---|---:|---|
| Precision | **0.9305** | Predicted detections are generally reliable. |
| Recall | **0.6000** | The detector still misses a meaningful number of plates. |
| mAP@50 | **0.6862** | Moderate unseen-data detection performance at IoU 0.50. |
| mAP@50–95 | **0.3832** | Localization quality decreases under stricter IoU thresholds. |
| Inference time | **36.62 ms/image** | Meets the project requirement of **<50 ms/image**. |
| GPU | **Tesla T4** | GPU-accelerated final evaluation. |

### Why the 60% Recall Matters

The detector's principal limitation is **missed detections**, not false detections.

With **25 annotated plates in the test set**, 60% recall corresponds approximately to detecting 15 ground-truth instances and missing about 10. Because the test set is small, a limited number of difficult samples has a large effect on the reported percentage.

The gap between strong validation performance and weaker test performance is evidence of a **generalization limitation**. It should be reported transparently rather than replacing the test result with the stronger validation number.


## 9. Save the selected checkpoint and training evidence to Google Drive

The selected model is copied to:

`My Drive/Number-Plate-Detection-Training-Results/models/best.pt`

The following evidence is also preserved:

- validation comparison table;
- final test summary;
- exact candidate hyperparameters;
- selected run directory;
- final test plots.

This folder can later be downloaded back into the VS Code project.

In [10]:
import json
import shutil

DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
(DRIVE_OUTPUT / "models").mkdir(exist_ok=True)
(DRIVE_OUTPUT / "reports").mkdir(exist_ok=True)

# Save final selected model without ever deleting the source checkpoint.
shutil.copy2(WINNER_PATH, DRIVE_OUTPUT / "models" / "best.pt")

comparison.to_csv(DRIVE_OUTPUT / "reports" / "validation_model_comparison.csv", index=False)
final_test.to_csv(DRIVE_OUTPUT / "reports" / "final_test_metrics.csv", index=False)

with (DRIVE_OUTPUT / "reports" / "training_candidates.json").open("w") as f:
    json.dump(CANDIDATES, f, indent=2)

selection = {
    "winner": WINNER_NAME,
    "winner_path": str(WINNER_PATH),
    "imgsz": WINNER_IMGSZ,
    "selection_metric": "validation mAP50-95",
    "gpu": GPU_NAME,
}
with (DRIVE_OUTPUT / "reports" / "model_selection.json").open("w") as f:
    json.dump(selection, f, indent=2)

# Copy final test plots.
test_run = PROJECT_DIR / "runs" / "test" / "final_test"
if test_run.exists():
    target = DRIVE_OUTPUT / "final_test_run"
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(test_run, target)

print("Saved all final training evidence to:")
print(DRIVE_OUTPUT)
print("\nFinal model:")
print(DRIVE_OUTPUT / "models" / "best.pt")

Saved all final training evidence to:
/content/drive/MyDrive/Number-Plate-Detection-Training-Results

Final model:
/content/drive/MyDrive/Number-Plate-Detection-Training-Results/models/best.pt


## 10. Final result summary

The cell below generates the final conclusion automatically from the validation comparison and untouched test-set results. This avoids manually replacing placeholder values and ensures the written conclusion matches the actual Colab run.

### Evaluation rule

Do not claim that retraining improved the detector unless the validation comparison actually shows an improvement over `existing_best`.

If the existing model wins, that is still a valid result: the retraining experiment was performed systematically and the stronger checkpoint was retained.


In [11]:
# ============================================================
# AUTOMATIC FINAL CONCLUSION
# ============================================================

result = final_test.iloc[0]

if "existing_best" in comparison["candidate"].values:
    baseline = comparison.loc[comparison["candidate"] == "existing_best"].iloc[0]
    delta = (winner["mAP50_95"] - baseline["mAP50_95"]) * 100

    if WINNER_NAME == "existing_best":
        comparison_statement = (
            "The existing detector remained the strongest validation model, so it was retained "
            "instead of replacing it with a weaker retrained checkpoint."
        )
    else:
        comparison_statement = (
            f"The selected retrained model improved validation mAP@50–95 by "
            f"{delta:+.2f} percentage points over the existing baseline."
        )
else:
    comparison_statement = (
        "No existing baseline checkpoint was supplied, so the final model was selected from "
        "the retrained candidates using validation mAP@50–95."
    )

final_conclusion = f"""
FINAL TRAINING CONCLUSION
-------------------------
YOLO26s detector training and model selection were performed on a Google Colab GPU using the frozen
90/13/12 train-validation-test split (115 images, 233 annotated number-plate boxes). Training explicitly
controlled input resolution, AdamW optimization, learning rate, automatic batch sizing, early stopping
and augmentation. The detector family used in this notebook is anchor-free, so manual anchor-box tuning
was not applicable.

Selected model: {WINNER_NAME}
Validation mAP@50–95: {winner['mAP50_95']:.4f}
Validation mAP@50:    {winner['mAP50']:.4f}

{comparison_statement}

The selected model was then evaluated once on the untouched test set and achieved:
Precision:      {result['precision']:.4f}
Recall:         {result['recall']:.4f}
mAP@50:         {result['mAP50']:.4f}
mAP@50–95:      {result['mAP50_95']:.4f}
Inference time: {result['inference_ms']:.2f} ms/image
GPU:            {result['gpu']}

Final checkpoint saved to:
{DRIVE_OUTPUT / 'models' / 'best.pt'}
""".strip()

print(final_conclusion)

summary_path = DRIVE_OUTPUT / "reports" / "FINAL_TRAINING_CONCLUSION.txt"
summary_path.write_text(final_conclusion)
print("\n✅ Final conclusion saved to:", summary_path)


FINAL TRAINING CONCLUSION
-------------------------
YOLO26s detector training and model selection were performed on a Google Colab GPU using the frozen
90/13/12 train-validation-test split (115 images, 233 annotated number-plate boxes). Training explicitly
controlled input resolution, AdamW optimization, learning rate, automatic batch sizing, early stopping
and augmentation. The detector family used in this notebook is anchor-free, so manual anchor-box tuning
was not applicable.

Selected model: existing_best
Validation mAP@50–95: 0.5216
Validation mAP@50:    0.9635

The existing detector remained the strongest validation model, so it was retained instead of replacing it with a weaker retrained checkpoint.

The selected model was then evaluated once on the untouched test set and achieved:
Precision:      0.9305
Recall:         0.6000
mAP@50:         0.6862
mAP@50–95:      0.3832
Inference time: 36.62 ms/image
GPU:            Tesla T4

Final checkpoint saved to:
/content/drive/MyDrive/N

## 11. Supplementary Model-Capacity Experiment — YOLO26m @ 832

The primary YOLO26s experiments showed that higher input resolution alone did not improve performance. Because the final selected model still produced only **60% recall on the untouched test set**, one additional experiment was performed to test whether a **larger detector** could improve the model's ability to find plates.

### Experimental Question

> Does increasing capacity from YOLO26s to YOLO26m improve validation performance enough to justify replacing the existing checkpoint?

To answer this without contaminating the test set:

1. the same frozen dataset is restored;
2. the existing `best.pt` baseline is re-evaluated on validation;
3. **YOLO26m @ 832 px** is trained using the same controlled settings;
4. YOLO26m is compared against the baseline on validation;
5. test evaluation is allowed **only if YOLO26m wins validation**.

This supplementary run is an experiment, not a post-hoc attempt to search the test set for a better-looking result.


In [1]:
# ============================================================
# YOLO26m EXPERIMENT — MINIMAL RESTORE
# ============================================================

!pip -q install -U ultralytics pyyaml "pandas==2.2.3"

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from zipfile import ZipFile
import shutil
import yaml
import pandas as pd
import torch
import ultralytics

print("Ultralytics:", ultralytics.__version__)
print("Pandas      :", pd.__version__)
print("CUDA        :", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "GPU is not enabled. Go to Runtime → Change runtime type → T4 GPU."
)

print("GPU:", torch.cuda.get_device_name(0))


ZIP_PATH = Path("/content/drive/MyDrive/number_plate_training.zip")
PROJECT_DIR = Path("/content/number_plate_training")
TEMP_EXTRACT = Path("/content/_number_plate_training_extract")

assert ZIP_PATH.exists(), f"ZIP not found: {ZIP_PATH}"

# Clean only the temporary local experiment workspace.
for path in [PROJECT_DIR, TEMP_EXTRACT]:
    if path.exists():
        shutil.rmtree(path)

TEMP_EXTRACT.mkdir(parents=True, exist_ok=True)

with ZipFile(ZIP_PATH, "r") as z:
    z.extractall(TEMP_EXTRACT)

# Support either ZIP structure.
nested_root = TEMP_EXTRACT / "number_plate_training"

if (nested_root / "data" / "processed").exists():
    extracted_root = nested_root
elif (TEMP_EXTRACT / "data" / "processed").exists():
    extracted_root = TEMP_EXTRACT
else:
    raise FileNotFoundError(
        "Could not find data/processed inside number_plate_training.zip."
    )

shutil.copytree(extracted_root, PROJECT_DIR)
shutil.rmtree(TEMP_EXTRACT)

DATA_DIR = PROJECT_DIR / "data" / "processed"
MODELS_DIR = PROJECT_DIR / "models"
CONFIG_DIR = PROJECT_DIR / "configs"

EXISTING_MODEL = MODELS_DIR / "best.pt"

assert DATA_DIR.exists(), DATA_DIR
assert EXISTING_MODEL.exists(), (
    "Existing models/best.pt is missing from the ZIP."
)

CONFIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_YAML = CONFIG_DIR / "data_colab.yaml"

data_yaml = {
    "path": str(DATA_DIR),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": {0: "number_plate"},
}

with DATA_YAML.open("w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print("\n✅ Experiment workspace restored")
print("Dataset :", DATA_DIR)
print("Baseline:", EXISTING_MODEL)
print("YAML    :", DATA_YAML)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.6 MB/s eta 0:00:00
Mounted at /content/drive
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics: 8.4.138
Pandas      : 2.2.3
CUDA        : True
GPU: Tesla T4

✅ Experiment workspace restored
Dataset : /content/number_plate_training/data/processed
Baseline: /content/number_plate_training/models/best.pt
YAML    : /content/number_plate_training/configs/data_colab.yaml


In [2]:
# ============================================================
# VERIFY FROZEN DATASET
# ============================================================

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

rows = []

for split in ["train", "val", "test"]:
    image_dir = DATA_DIR / split / "images"
    label_dir = DATA_DIR / split / "labels"

    images = [
        p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ]

    labels = list(label_dir.glob("*.txt"))

    boxes = sum(
        sum(1 for line in lp.read_text().splitlines() if line.strip())
        for lp in labels
    )

    rows.append({
        "split": split,
        "images": len(images),
        "labels": len(labels),
        "boxes": boxes
    })

dataset_check = pd.DataFrame(rows)
display(dataset_check)

assert dataset_check["images"].sum() == 115
assert dataset_check["boxes"].sum() == 233

split_check = dataset_check.set_index("split")

assert split_check.loc["train", "images"] == 90
assert split_check.loc["val", "images"] == 13
assert split_check.loc["test", "images"] == 12

print("✅ Frozen 90/13/12 dataset verified.")

,split,images,labels,boxes
0,train,90,90,185
1,val,13,13,23
2,test,12,12,25


✅ Frozen 90/13/12 dataset verified.


In [3]:
# ============================================================
# BASELINE VALIDATION
# ============================================================

from ultralytics import YOLO

baseline_model = YOLO(str(EXISTING_MODEL))

baseline_val = baseline_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=832,
    batch=4,
    device=0,
    plots=False,
    verbose=False,
)

baseline_metrics = {
    "candidate": "existing_best",
    "precision": float(baseline_val.box.mp),
    "recall": float(baseline_val.box.mr),
    "mAP50": float(baseline_val.box.map50),
    "mAP50_95": float(baseline_val.box.map),
    "inference_ms": float(
        (getattr(baseline_val, "speed", {}) or {}).get(
            "inference", float("nan")
        )
    ),
}

display(pd.DataFrame([baseline_metrics]))

print(
    f"Baseline validation mAP50-95: "
    f"{baseline_metrics['mAP50_95']:.4f}"
)

Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s summary (fused): 122 layers, 9,465,567 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3743.8±922.3 MB/s, size: 1921.6 KB)
val: Scanning /content/number_plate_training/data/processed/val/labels... 13 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 13/13 307.6it/s 0.0s
val: New cache created: /content/number_plate_training/data/processed/val/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.5it/s 1.6s
                   all         13         23      0.991       0.87      0.963      0.522
Speed: 1.6ms preprocess, 34.0ms inference, 0.0ms loss, 0.2ms postprocess per image


,candidate,precision,recall,mAP50,mAP50_95,inference_ms
0,existing_best,0.991076,0.869565,0.963474,0.521626,34.032644


Baseline validation mAP50-95: 0.5216


In [4]:
# ============================================================
# YOLO26m @ 832 EXPERIMENT
# ============================================================

import random
import numpy as np
import os

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

RUN_ROOT = PROJECT_DIR / "runs" / "detect"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

EXPERIMENT_NAME = "yolo26m_832_experiment"

model = YOLO("yolo26m.pt")

model.train(
    data=str(DATA_YAML),

    # Model/input
    imgsz=832,
    epochs=120,
    batch=-1,

    # Optimization
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,

    # Hardware/reproducibility
    device=0,
    workers=2,
    seed=SEED,
    deterministic=True,

    # Early stopping
    patience=25,

    # Output
    project=str(RUN_ROOT),
    name=EXPERIMENT_NAME,
    exist_ok=True,
    plots=True,

    # Augmentation — same as previous controlled experiment
    degrees=5.0,
    translate=0.10,
    scale=0.40,
    fliplr=0.50,
    hsv_h=0.015,
    hsv_s=0.60,
    hsv_v=0.35,
    mosaic=0.80,
    mixup=0.05,
    erasing=0.10,
)

M_MODEL_PATH = (
    RUN_ROOT /
    EXPERIMENT_NAME /
    "weights" /
    "best.pt"
)

assert M_MODEL_PATH.exists()

print("\n✅ YOLO26m training complete")
print("Checkpoint:", M_MODEL_PATH)

Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/number_plate_training/configs/data_colab.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.1, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.6, hsv_v=0.35, imgsz=832, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=yolo26m.pt, momentum=0.937, mosaic=0.8, multi_scale=0.0, name=yolo

In [5]:
# ============================================================
# VALIDATION COMPARISON
# ============================================================

m_model = YOLO(str(M_MODEL_PATH))

m_val = m_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=832,
    batch=4,
    device=0,
    plots=False,
    verbose=False,
)

m_speed = getattr(m_val, "speed", {}) or {}

m_metrics = {
    "candidate": "yolo26m_832",
    "precision": float(m_val.box.mp),
    "recall": float(m_val.box.mr),
    "mAP50": float(m_val.box.map50),
    "mAP50_95": float(m_val.box.map),
    "inference_ms": float(
        m_speed.get("inference", float("nan"))
    ),
}

comparison = pd.DataFrame([
    baseline_metrics,
    m_metrics
]).sort_values(
    "mAP50_95",
    ascending=False
).reset_index(drop=True)

display(
    comparison.style.format({
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "mAP50": "{:.4f}",
        "mAP50_95": "{:.4f}",
        "inference_ms": "{:.2f}",
    })
)

baseline_score = baseline_metrics["mAP50_95"]
m_score = m_metrics["mAP50_95"]

delta = m_score - baseline_score

print("\nExisting best mAP50-95 :", f"{baseline_score:.4f}")
print("YOLO26m mAP50-95      :", f"{m_score:.4f}")
print("Difference             :", f"{delta:+.4f}")

if m_score > baseline_score:
    print("\n✅ YOLO26m beats the existing model on validation.")
    print("Proceed to TEST evaluation.")
else:
    print("\n❌ YOLO26m does NOT beat the existing model.")
    print("Keep the existing best.pt.")
    print("Do NOT use the test set to rescue/select YOLO26m.")

Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26m summary (fused): 132 layers, 20,350,223 parameters, 0 gradients, 68.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3951.8±875.5 MB/s, size: 2125.4 KB)
val: Scanning /content/number_plate_training/data/processed/val/labels.cache... 13 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 13/13 3.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.0it/s 2.0s
                   all         13         23      0.804       0.89       0.88      0.517
Speed: 3.0ms preprocess, 46.5ms inference, 0.0ms loss, 0.6ms postprocess per image


,candidate,precision,recall,mAP50,mAP50_95,inference_ms
0,existing_best,0.9911,0.8696,0.9635,0.5216,34.03
1,yolo26m_832,0.8037,0.8903,0.8798,0.5171,46.49



Existing best mAP50-95 : 0.5216
YOLO26m mAP50-95      : 0.5171
Difference             : -0.0045

❌ YOLO26m does NOT beat the existing model.
Keep the existing best.pt.
Do NOT use the test set to rescue/select YOLO26m.


### YOLO26m Experiment Result

The higher-capacity YOLO26m model produced the following validation comparison:

| Metric | Existing `best.pt` | YOLO26m @ 832 | Outcome |
|---|---:|---:|---|
| Precision | **0.9911** | 0.8037 | Existing model clearly stronger |
| Recall | 0.8696 | **0.8903** | YOLO26m improved by ~2.1 percentage points |
| mAP@50 | **0.9635** | 0.8798 | Existing model stronger |
| mAP@50–95 | **0.5216** | 0.5171 | Existing model stronger |
| Inference | **34.03 ms/image** | 46.49 ms/image | Existing model faster |

YOLO26m increased validation recall from **86.96% to 89.03%**, showing that additional model capacity helped find slightly more validation plates.

However, this came with a substantial precision reduction from **99.11% to 80.37%**, lower mAP@50, slightly lower mAP@50–95, and slower inference.

The predefined selection metric was:

**Validation mAP@50–95**

- Existing checkpoint: **0.5216**
- YOLO26m @ 832: **0.5171**
- Difference: **−0.0045**

Therefore, YOLO26m **did not beat the incumbent** and was not evaluated on the test set.

This is intentional. Testing YOLO26m after it lost validation would allow the test set to influence model selection and weaken the credibility of the final evaluation.

### Conclusion from the Capacity Experiment

Increasing detector capacity did **not** solve the problem sufficiently.

The experiment suggests that the project's remaining recall/generalization limitation is unlikely to be fixed simply by moving to a larger YOLO model. With only **90 training images**, additional and more diverse labeled data, operating-point optimization, and downstream pipeline handling are more defensible next steps than repeatedly increasing model size.


## 12. Consolidated Detector Results for Final Presentation

### Experiments Completed

| Experiment | Purpose | Result |
|---|---|---|
| Existing `best.pt` | Establish incumbent baseline | **Final winner** |
| YOLO26s @ 832 | Controlled retraining at higher resolution | Did not beat baseline |
| YOLO26s @ 1024 | Test whether greater resolution improves plate localization | Did not beat baseline; no benefit from 1024 px |
| YOLO26m @ 832 | Test whether greater model capacity improves recall/generalization | Recall improved slightly, but overall validation quality declined |

### Final Selected Detector

**Model:** Existing `best.pt`

**Validation**
- Precision: **99.11%**
- Recall: **86.96%**
- mAP@50: **96.35%**
- mAP@50–95: **52.16%**

**Untouched Test**
- Precision: **93.05%**
- Recall: **60.00%**
- mAP@50: **68.62%**
- mAP@50–95: **38.32%**
- Inference time: **36.62 ms/image**
- GPU: **Tesla T4**

### Project Target Assessment

| Project Requirement | Evidence | Status |
|---|---|---|
| 95%+ detection accuracy | Validation mAP@50 = 96.35%, but test mAP@50 = 68.62% and test recall = 60% | **Not demonstrated consistently on unseen data** |
| OCR accuracy >90% | Not measured in this detector-training notebook | **Evaluated separately in OCR notebook** |
| Inference <50 ms/image | 36.62 ms/image | **Achieved** |
| Handle lighting / backgrounds / variability | Real-world variation is represented, but missed test detections remain | **Partially achieved** |
| Streamlit deployment | Outside detector training | **Demonstrated separately in application/deployment stage** |

### Final Technical Interpretation

The training process did not simply select the newest or largest model. Multiple controlled experiments were conducted and the incumbent checkpoint was retained because it remained the strongest model under the predefined validation criterion.

The **95%+ project detection target should not be claimed as achieved on unseen data**. Although validation mAP@50 reached 96.35%, the untouched test set produced **68.62% mAP@50 and 60% recall**.

At the same time, the detector achieved two meaningful strengths:

- **93.05% test precision**, indicating reliable predictions when detections are made;
- **36.62 ms/image inference**, satisfying the real-time performance requirement.

The primary remaining weakness is recall/generalization. The YOLO26m experiment showed that increasing model capacity improved validation recall only marginally and reduced overall validation quality, so further architecture scaling was not justified.

### Presentation-Ready Conclusion

> The final detector was selected through controlled, validation-based anti-regression testing. YOLO26s models at 832 and 1024 pixels and a higher-capacity YOLO26m model were evaluated against the existing checkpoint. None improved validation mAP@50–95, so the existing model was retained. On the untouched test set it achieved 93.05% precision, 60% recall, 68.62% mAP@50 and 36.62 ms/image inference. The real-time target was achieved, while the original 95%+ detection target was not consistently demonstrated on unseen data. The remaining limitation is primarily missed detections/generalization rather than prediction precision, and this is carried forward transparently into operating-point and OCR evaluation.
